# Sesión 6 — MUBD

## Clasificación supervisada (I)

Estudiamos **KNN** para anticipar bajas de clientes y **Naive Bayes** para clasificar productos.
Seguimos el recorrido de la sesión: preparación de datos, predicción, probabilidades y selección de variables.
Las prácticas 6-1 y 6-2 quedan para resolver.

En VS Code, selecciona un kernel de Python y ejecuta las celdas en orden con **Shift + Enter**.
Abre **Variables** para explorar las tablas y **Esquema / Outline** para navegar por la sesión.
Los controles se activan al ejecutar sus celdas; puedes volver a ellos después de avanzar.
Usa **Reiniciar** y **Ejecutar todo** para repetir el análisis desde cero.
[Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

### Preparación

Limpiamos las variables de la sesión y cargamos las librerías.
Si falta alguna en el kernel seleccionado, descomenta y ejecuta la instalación una vez.
Las funciones de apoyo están incluidas en celdas que puedes desplegar.

In [ ]:
%reset -f

In [ ]:
# %pip install numpy pandas matplotlib scipy scikit-learn ipywidgets threadpoolctl ipykernel

In [ ]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy import stats
from scipy.signal import fftconvolve
from scipy.special import logsumexp
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})
pd.set_option("display.max_columns", 30)
SEED = 12345

In [ ]:
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos" / "edw.csv").is_file():
        DATA_PATH = carpeta / "datos"
        break
else:
    raise FileNotFoundError("Abre el notebook dentro de la carpeta LaSalle_Estadistica.")

DATA_PATH

Las matrices de confusión tendrán **predicciones en las filas** y **clases reales en las columnas**.
La proporción de aciertos es el número de observaciones bien clasificadas dividido por el total.
Si permitimos rechazos, distinguimos el acierto entre casos clasificados y la cobertura.

In [ ]:
def tabla_confusion(reales, predichos, clases=None, incluir_rechazos=False):
    reales = np.asarray(reales)
    predichos = np.asarray(predichos)
    clases = np.unique(reales) if clases is None else np.asarray(clases)
    tabla = pd.crosstab(pd.Series(predichos, name="Predicción"),
                        pd.Series(reales, name="Real"))
    tabla = tabla.reindex(index=clases, columns=clases, fill_value=0)
    if incluir_rechazos:
        tabla.loc["Sin decisión"] = [np.sum(pd.isna(predichos) & (reales == c)) for c in clases]
    return tabla.rename_axis(index="Predicción", columns="Real")


def evaluar(reales, predichos, clases=None):
    reales, predichos = np.asarray(reales), np.asarray(predichos)
    aceptadas = ~pd.isna(predichos)
    aciertos = aceptadas & (reales == predichos)
    return pd.Series({"Acierto entre clasificadas": aciertos.sum() / aceptadas.sum() if aceptadas.any() else np.nan,
                      "Cobertura": aceptadas.mean(),
                      "Acierto sobre todo el test": aciertos.mean()})

## 1. KNN: bajas de clientes

Cada cliente se clasifica según los votos de sus vecinos más próximos, usando distancia euclídea.
Comenzamos con las variables numéricas en sus unidades originales.

### Diccionario de variables

| Variable | Contenido |
| --- | --- |
| `age`, `annualincome` | Edad e ingresos anuales |
| `calldroprate`, `callfailurerate` | Tasas de llamadas perdidas y cortadas |
| `callingnum`, `customerid` | Teléfono e identificador del cliente |
| `customersuspended` | Indicador de suspensión del cliente |
| `education`, `gender`, `maritalstatus` | Formación, sexo y estado civil |
| `homeowner` | Propiedad de la vivienda |
| `monthlybilledamount` | Factura mensual |
| `noadditionallines` | Campo sin información útil en este archivo |
| `numberofcomplaints`, `numberofmonthunpaid` | Quejas y meses sin pagar |
| `numdayscontractequipmentplanexpiring` | Días hasta el vencimiento del contrato |
| `occupation` | Ocupación |
| `penaltytoswitch` | Penalización por cambio de línea |
| `state` | Estado de Estados Unidos |
| `totalminsusedinlastmonth` | Minutos utilizados el último mes |
| `unpaidbalance` | Saldo pendiente de pago |
| `usesinternetservice`, `usesvoiceservice` | Servicios de internet y voz contratados |
| `percentagecalloutsidenetwork` | Proporción de llamadas fuera de la red |
| `totalcallduration`, `avgcallduration` | Duración total y media de las llamadas |
| `churn` | Baja del cliente: 0 o 1 |
| `year`, `month` | Año y mes |

### Leer e inspeccionar los datos

In [ ]:
d0 = pd.read_csv(DATA_PATH / "edw.csv")
d0.shape
d0.head()

In [ ]:
d0.describe(include="all")

### Eliminar variables irrelevantes y transformar categorías

Retiramos identificadores y campos sin interés para este ejercicio.
La formación se codifica con **High School or below** como primer nivel.
La ocupación genera dos indicadores; **Others** queda como categoría de referencia.

In [ ]:
variables_retiradas = ["customerid", "noadditionallines", "state", "year", "month", "callingnum"]
d = d0.drop(columns=variables_retiradas).copy()
niveles_educacion = ["High School or below", "Bachelor or equivalent", "Master or equivalent", "PhD or equivalent"]
d["education"] = pd.Categorical(d["education"], categories=niveles_educacion, ordered=True)
d["occupationTech"] = d["occupation"].eq("Technology Related Job").astype(int)
d["occupationNonTech"] = d["occupation"].eq("Non-technology Related Job").astype(int)
d = d.drop(columns="occupation")
d = d[[c for c in d.columns if c != "churn"] + ["churn"]]
d.head()

Todas las entradas de KNN deben ser numéricas.
Asignamos códigos desde 1 a las categorías, conservando el orden de formación indicado.
`codificacion` permite consultar qué representa cada código.

In [ ]:
codificacion = {}
for variable in d.select_dtypes(include=["object", "category"]).columns:
    niveles = niveles_educacion if variable == "education" else sorted(d[variable].unique())
    codificacion[variable] = dict(zip(niveles, range(1, len(niveles)+1)))
    d[variable] = pd.Categorical(d[variable], categories=niveles).codes + 1

codificacion
d.dtypes

### Dividir la muestra

Cada fila entra en entrenamiento con probabilidad 0,7. El tamaño final puede variar ligeramente.
La semilla fija hace reproducible la partición dentro de Python.

In [ ]:
p_train = 0.7
rng_knn = np.random.default_rng(SEED)
train_sel_knn = rng_knn.choice([False, True], size=len(d), p=[1-p_train, p_train])
train_knn = d.loc[train_sel_knn].copy()
test_knn = d.loc[~train_sel_knn].copy()
X_train_knn = train_knn.drop(columns="churn")
X_test_knn = test_knn.drop(columns="churn")
y_train_knn = train_knn["churn"]
y_test_knn = test_knn["churn"]
pd.Series({"Entrenamiento": len(train_knn), "Test": len(test_knn)})

### Herramientas de vecinos

Guardamos las distancias y los vecinos para cambiar `k` sin repetir toda la búsqueda.
El voto uniforme incorpora los empates en la distancia del vecino límite; los empates de voto se
resuelven con una semilla fija. El kernel óptimo asigna pesos decrecientes según el rango del vecino.

In [ ]:
class Vecinos:
    """Vecinos euclídeos reutilizables; voto uniforme o ponderado por kernel."""

    def __init__(self, X, y, consulta=None, max_k=21, estandarizar=False):
        self.dimension = np.asarray(X).shape[1]
        self.clases, self.y = np.unique(y, return_inverse=True)
        self.indice = X.index.copy() if consulta is None else consulta.index.copy()
        self.escala = np.asarray(X).std(axis=0, ddof=1) if estandarizar else np.ones(self.dimension)
        self.escala[self.escala == 0] = 1
        self.X = np.asarray(X, dtype=float) / self.escala
        self.consulta = None if consulta is None else np.asarray(consulta, dtype=float) / self.escala
        self.max_k = min(max_k, len(X) - 1)
        self.buscador = NearestNeighbors(n_neighbors=self.max_k + 1, algorithm="auto", n_jobs=1).fit(self.X)
        with threadpool_limits(limits=1):
            self.distancias, self.indices = self.buscador.kneighbors(self.consulta)

    def probabilidades(self, k=1, kernel="uniforme"):
        if not 1 <= k <= self.max_k:
            raise ValueError(f"k debe estar entre 1 y {self.max_k}.")
        indices = self.indices
        if kernel == "uniforme":
            radio = self.distancias[:, k-1] * np.sqrt(1 + 1e-4)
            pesos = (self.distancias <= radio[:, None]).astype(float)
        else:
            indices = indices[:, :k]
            if kernel == "óptimo":
                rango = np.arange(1, k+1, dtype=float)
                exponente = 1 + 2 / self.dimension
                pesos_rango = (1 + self.dimension/2 - self.dimension *
                               (rango**exponente - (rango-1)**exponente) / (2*k**(2/self.dimension))) / k
                pesos = np.broadcast_to(pesos_rango, indices.shape)
            else:
                ancho = np.maximum(self.distancias[:, k], 1e-6)
                u = np.clip(self.distancias[:, :k] / ancho[:, None], 1e-6, 1-1e-6)
                if kernel == "triangular":
                    pesos = 1-u
                elif kernel == "epanechnikov":
                    pesos = 0.75*(1-u*u)
                else:
                    raise ValueError("Kernel desconocido.")
        votos = np.column_stack([(pesos * (self.y[indices] == j)).sum(axis=1)
                                 for j in range(len(self.clases))])
        if kernel == "uniforme":
            # Si un empate rebasa la caché, incluimos todos los vecinos del radio.
            desbordadas = np.flatnonzero(pesos[:, -1] > 0)
            for fila in desbordadas:
                punto = self.X[fila] if self.consulta is None else self.consulta[fila]
                cercanos = self.buscador.radius_neighbors(punto.reshape(1, -1),
                           radius=np.nextafter(radio[fila], np.inf), return_distance=False)[0]
                if self.consulta is None:
                    cercanos = cercanos[cercanos != fila]
                votos[fila] = np.bincount(self.y[cercanos], minlength=len(self.clases))
        self.ultimos_votos = votos
        return pd.DataFrame(votos / votos.sum(axis=1, keepdims=True),
                            columns=self.clases, index=self.indice)

    def predecir(self, k=1, kernel="uniforme", minimo=0):
        probabilidades = self.probabilidades(k, kernel)
        valores = probabilidades.to_numpy()
        ganadora = valores.argmax(axis=1)
        if kernel == "uniforme":
            rng = np.random.default_rng(SEED)
            for fila in np.flatnonzero((valores == valores.max(axis=1, keepdims=True)).sum(axis=1) > 1):
                ganadora[fila] = rng.choice(np.flatnonzero(valores[fila] == valores[fila].max()))
        predichos = pd.Series(self.clases[ganadora], index=self.indice, name="Predicción")
        if minimo and kernel == "uniforme":
            extras = self.ultimos_votos.sum(axis=1) - k
            rechazo = self.ultimos_votos.max(axis=1) < minimo + extras
            predichos = predichos.where(~rechazo)
        return predichos


def barrido_vecinos(ajuste, reales, valores_k, kernel="uniforme"):
    return pd.Series({k: np.mean(ajuste.predecir(k, kernel).to_numpy() == np.asarray(reales))
                      for k in valores_k}, name="Proporción de aciertos").rename_axis("k").to_frame()


def dibujar_barrido(resultados, titulo):
    fig, ax = plt.subplots()
    ax.plot(resultados.index, resultados.iloc[:, 0], "o-")
    ax.set(xlabel="Número de vecinos (k)", ylabel="Proporción de aciertos", title=titulo,
           xticks=resultados.index)
    ax.grid(alpha=0.2)
    plt.show()

### 1-NN: entrenamiento y test

In [ ]:
vecinos_originales = Vecinos(X_train_knn, y_train_knn, X_test_knn)
knn1 = vecinos_originales.predecir(k=1)
t_knn1 = tabla_confusion(y_test_knn, knn1)
t_knn1
np.mean(knn1.to_numpy() == y_test_knn.to_numpy())

### 1-NN: validación dejando una observación fuera

Para predecir cada fila del conjunto completo excluimos esa misma fila de sus vecinos.
Así obtenemos la validación *leave-one-out* sin ajustar miles de modelos por separado.

In [ ]:
vecinos_loo = Vecinos(d.drop(columns="churn"), d["churn"], max_k=1)
knn_loo = vecinos_loo.predecir(k=1)
t_loo = tabla_confusion(d["churn"], knn_loo)
t_loo
np.mean(knn_loo.to_numpy() == d["churn"].to_numpy())

### Regla de la clase mayoritaria

In [ ]:
frecuencias_test = y_test_knn.value_counts().sort_index()
frecuencias_test
frecuencias_test.max() / frecuencias_test.sum()

### Número de vecinos

Probamos los valores impares de 1 a 21. La curva representa el acierto observado en test.

In [ ]:
K = np.arange(1, 22, 2)
aciertos_originales = barrido_vecinos(vecinos_originales, y_test_knn, K)
dibujar_barrido(aciertos_originales, "KNN con las variables originales")
aciertos_originales

#### Explorar una predicción

Cambia el número de vecinos y la posición del cliente en test.
La tabla identifica sus vecinos de entrenamiento, sus clases y sus distancias.

In [ ]:
def explorar_knn(k, cliente):
    prediccion = vecinos_originales.predecir(k)
    display(tabla_confusion(y_test_knn, prediccion), evaluar(y_test_knn, prediccion))
    fila = int(cliente)
    display(pd.Series({"Índice del cliente": X_test_knn.index[fila],
                       "Clase real": y_test_knn.iloc[fila], "Predicción": prediccion.iloc[fila]}))
    posiciones = vecinos_originales.indices[fila, :k]
    detalle = pd.DataFrame({"Índice de entrenamiento": X_train_knn.index[posiciones],
                            "Distancia": vecinos_originales.distancias[fila, :k],
                            "churn": y_train_knn.iloc[posiciones].to_numpy()})
    display(detalle)

control_knn = widgets.interactive(explorar_knn,
    k=widgets.IntSlider(value=1, min=1, max=21, step=2, description="Vecinos", continuous_update=False),
    cliente=widgets.IntSlider(value=0, min=0, max=len(test_knn)-1, description="Cliente", continuous_update=False))
display(control_knn)

### ACP antes de KNN

Calculamos componentes principales sobre las variables en sus unidades originales y conservamos
solo la primera componente. Se mantiene la misma partición de filas.
El ACP de este recorrido se calcula sobre el conjunto completo: sus resultados son exploratorios.
Para evaluar una solución final, ajusta también la transformación exclusivamente con entrenamiento.

In [ ]:
acp_knn = PCA(svd_solver="full")
componentes_knn = acp_knn.fit_transform(d.drop(columns="churn"))
varianza_componentes = pd.Series(acp_knn.explained_variance_,
                                index=np.arange(1, componentes_knn.shape[1]+1), name="Varianza")
fig, ax = plt.subplots()
ax.plot(varianza_componentes.index, varianza_componentes, "o-")
ax.set(xlabel="Componente principal", ylabel="Varianza", title="Scree plot")
plt.show()
pd.DataFrame({"Varianza": varianza_componentes, "Proporción explicada": acp_knn.explained_variance_ratio_})

In [ ]:
cp1_knn = pd.DataFrame({"CP1": componentes_knn[:, 0]}, index=d.index)
train_cp1 = cp1_knn.loc[train_sel_knn]
test_cp1 = cp1_knn.loc[~train_sel_knn]
vecinos_cp1 = Vecinos(train_cp1, y_train_knn, test_cp1)
aciertos_cp1 = barrido_vecinos(vecinos_cp1, y_test_knn, K)
dibujar_barrido(aciertos_cp1, "KNN con la primera componente principal")
aciertos_cp1

In [ ]:
knn_cp1 = vecinos_cp1.predecir(k=1)
tabla_confusion(y_test_knn, knn_cp1)
np.mean(knn_cp1.to_numpy() == y_test_knn.to_numpy())

#### Explorar el voto en una dimensión

La altura separa las clases para facilitar la lectura; la distancia utiliza únicamente CP1.
Los puntos representan una muestra reproducible de entrenamiento. La matriz usa todo el test.

In [ ]:
def explorar_cp1(k):
    prediccion = vecinos_cp1.predecir(k)
    display(tabla_confusion(y_test_knn, prediccion), evaluar(y_test_knn, prediccion))
    seleccion = np.random.default_rng(SEED).choice(len(train_cp1), size=min(1000, len(train_cp1)), replace=False)
    fig, ax = plt.subplots(figsize=(9, 3))
    for clase in np.unique(y_train_knn):
        posiciones = seleccion[y_train_knn.iloc[seleccion].to_numpy() == clase]
        ax.scatter(train_cp1.iloc[posiciones, 0], np.full(len(posiciones), clase), s=15, alpha=0.35, label=f"churn {clase}")
    ax.set(xlabel="CP1", ylabel="Clase", yticks=np.unique(y_train_knn), title=f"Entrenamiento · k={k}")
    ax.legend()
    plt.show()

control_cp1 = widgets.interactive(explorar_cp1,
    k=widgets.IntSlider(value=1, min=1, max=21, step=2, description="Vecinos", continuous_update=False))
display(control_cp1)

### Exigir un mínimo de votos

Con `k=2` y mínimo de 2 votos, los dos vecinos deben coincidir para dar una decisión.
Los casos dudosos se muestran como **Sin decisión**. Al aumentar el mínimo puede bajar la cobertura.
Si la distancia límite añade vecinos empatados, el mínimo exigido aumenta con esos vecinos adicionales.

In [ ]:
knn_con_rechazo = vecinos_cp1.predecir(k=2, minimo=2)
tabla_confusion(y_test_knn, knn_con_rechazo, incluir_rechazos=True)
evaluar(y_test_knn, knn_con_rechazo)

In [ ]:
def explorar_rechazo(k, minimo):
    if minimo > k:
        display("El mínimo de votos debe ser menor o igual que k.")
        return
    prediccion = vecinos_cp1.predecir(k=k, minimo=minimo)
    display(tabla_confusion(y_test_knn, prediccion, incluir_rechazos=True), evaluar(y_test_knn, prediccion))

control_rechazo = widgets.interactive(explorar_rechazo,
    k=widgets.IntSlider(value=2, min=1, max=21, description="Vecinos", continuous_update=False),
    minimo=widgets.IntSlider(value=2, min=0, max=21, description="Voto mínimo", continuous_update=False))
display(control_rechazo)

### KNN ponderado por kernel

El kernel óptimo utiliza el rango de cada vecino y el número de predictores para ponderar el voto.
En este bloque igualamos las desviaciones típicas utilizando las de entrenamiento.
Los controles permiten probar también pesos triangulares y de Epanechnikov.

In [ ]:
vecinos_kernel = Vecinos(X_train_knn, y_train_knn, X_test_knn, estandarizar=True)
kknn1 = vecinos_kernel.predecir(k=1, kernel="óptimo")
tabla_confusion(y_test_knn, kknn1)
np.mean(kknn1.to_numpy() == y_test_knn.to_numpy())

In [ ]:
aciertos_kernel = barrido_vecinos(vecinos_kernel, y_test_knn, K, kernel="óptimo")
dibujar_barrido(aciertos_kernel, "KNN ponderado · variables originales")
aciertos_kernel

#### Kernel con ACP

In [ ]:
vecinos_kernel_cp1 = Vecinos(train_cp1, y_train_knn, test_cp1, estandarizar=True)
kknn_cp1 = vecinos_kernel_cp1.predecir(k=2, kernel="óptimo")
tabla_confusion(y_test_knn, kknn_cp1)
np.mean(kknn_cp1.to_numpy() == y_test_knn.to_numpy())

In [ ]:
aciertos_kernel_cp1 = barrido_vecinos(vecinos_kernel_cp1, y_test_knn, K, kernel="óptimo")
dibujar_barrido(aciertos_kernel_cp1, "KNN ponderado · primera componente")
aciertos_kernel_cp1

In [ ]:
def explorar_kernel_knn(datos, kernel, k):
    ajuste = vecinos_kernel if datos == "Originales" else vecinos_kernel_cp1
    prediccion = ajuste.predecir(k=k, kernel=kernel)
    display(tabla_confusion(y_test_knn, prediccion), evaluar(y_test_knn, prediccion))
    display(ajuste.probabilidades(k=k, kernel=kernel).head().rename_axis(columns="Clase"))

control_kernel_knn = widgets.interactive(explorar_kernel_knn,
    datos=widgets.Dropdown(options=["Originales", "CP1"], description="Datos"),
    kernel=widgets.Dropdown(options=["óptimo", "triangular", "epanechnikov"], description="Kernel"),
    k=widgets.IntSlider(value=1, min=1, max=21, description="Vecinos", continuous_update=False))
display(control_kernel_knn)

## Ejercicio 6-1

1. Lee `bank-additional-full.csv`, separado por punto y coma, desde `DATA_PATH`.
2. Depura los datos: ordena las categorías ordinales, crea dummies para las nominales,
   transforma las entradas a valores numéricos y elimina `pdays`.

Puedes usar la siguiente función de apoyo. Para una variable ordinal debes indicar `orden`.
En las nominales se generan indicadores para todas las categorías salvo la última, que queda de referencia.

In [ ]:
def to_dummy(x, ordinal=False, orden=None):
    """Códigos ordinales desde 1 o dummies con la última categoría de referencia."""
    serie = pd.Series(x)
    if ordinal:
        if orden is None:
            raise ValueError("Indica el orden de las categorías.")
        codigos = pd.Series(pd.Categorical(serie, categories=orden, ordered=True).codes,
                            index=serie.index, name=serie.name)
        return (codigos + 1).where(codigos >= 0)
    categorias = sorted(serie.dropna().unique())
    return pd.DataFrame({f"{categoria}.dummy": (serie == categoria).astype(int)
                         for categoria in categorias[:-1]}, index=serie.index)

In [ ]:
# 1. Leer el archivo bank-additional-full.csv.

In [ ]:
# 2. Preparar las variables y retirar pdays.

3. Selecciona una submuestra reproducible de 10 000 observaciones y divídela en entrenamiento y test.
4. Estudia qué número de vecinos ofrece una proporción de aciertos adecuada.

In [ ]:
# 3. Submuestra de 10 000 observaciones y partición.

In [ ]:
# 4. Estudiar el número de vecinos.

## 2. Naive Bayes: clasificación de productos

Usamos objetos propios de este bloque para mantener disponibles los controles de KNN.

| Variable | Contenido |
| --- | --- |
| `id` | Identificador del producto |
| `feat_1` … `feat_93` | Características del producto |
| `target` | Clase de producto |

### Leer e inspeccionar los datos

In [ ]:
datos = pd.read_csv(DATA_PATH / "products.csv", sep=";", decimal=",")
datos.shape
datos.head()

In [ ]:
datos.describe(include="all")

In [ ]:
missings_por_columna = datos.isna().sum()
missings_por_columna.value_counts().sort_index().rename_axis("Valores ausentes").to_frame("Número de columnas")

### Premisa de independencia

Naive Bayes modela los predictores como independientes **condicionados a la clase**.
El gráfico siguiente resume las correlaciones marginales entre pares de características.
Una correlación baja, por sí sola, no establece independencia condicional.

In [ ]:
predictores_nb = datos.columns.difference(["id", "target"], sort=False)
cor_productos = datos[predictores_nb].corr()
cor_pares = cor_productos.to_numpy()[np.triu_indices(len(predictores_nb), k=1)]
intervalos_cor = pd.cut(cor_pares, bins=np.linspace(-1, 1, 21), include_lowest=True)
tabla_cor = pd.Series(intervalos_cor).value_counts(sort=False).rename("Pares")
tabla_cor

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(np.arange(len(tabla_cor)), tabla_cor.to_numpy())
ax.set(xticks=np.arange(len(tabla_cor)), xticklabels=[str(x) for x in tabla_cor.index],
       xlabel="Intervalo de correlación", ylabel="Pares de variables", title="Correlaciones entre características")
plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
fig.tight_layout()
plt.show()

### Dividir la muestra

Retiramos `id` y volvemos a seleccionar filas con probabilidad 0,7 para entrenamiento.

In [ ]:
d_productos = datos.drop(columns="id")
rng_nb = np.random.default_rng(SEED)
train_sel_nb = rng_nb.choice([False, True], size=len(d_productos), p=[0.3, 0.7])
train_nb = d_productos.loc[train_sel_nb].copy()
test_nb = d_productos.loc[~train_sel_nb].copy()
X_train_nb, y_train_nb = train_nb.drop(columns="target"), train_nb["target"]
X_test_nb, y_test_nb = test_nb.drop(columns="target"), test_nb["target"]
pd.Series({"Entrenamiento": len(train_nb), "Test": len(test_nb)})

### Ajustar Naive Bayes gaussiano

Estimamos una media y una desviación típica muestral para cada predictor dentro de cada clase.
Las probabilidades a priori se obtienen de las frecuencias de las clases.
Calculamos los productos de densidades mediante sumas de logaritmos.
Una desviación nula o una densidad que se redondea a cero se sustituye por 0,001.

In [ ]:
class BayesGaussiano:
    """Naive Bayes numérico con medias y desviaciones muestrales por clase."""

    def __init__(self, laplace=0, umbral=0.001):
        self.laplace, self.umbral = laplace, umbral

    def fit(self, X, y):
        self.variables = list(X.columns)
        self.clases = np.unique(y)
        self.recuentos = pd.Series(y).value_counts().reindex(self.clases)
        self.prior = self.recuentos / self.recuentos.sum()
        grupos = [np.asarray(X, dtype=float)[np.asarray(y) == clase] for clase in self.clases]
        self.medias = pd.DataFrame([grupo.mean(axis=0) for grupo in grupos],
                                  index=self.clases, columns=self.variables)
        self.desviaciones = pd.DataFrame([grupo.std(axis=0, ddof=1) for grupo in grupos],
                                        index=self.clases, columns=self.variables)
        self.tablas = {v: pd.DataFrame({"Media": self.medias[v], "Desviación": self.desviaciones[v]})
                       for v in self.variables}
        return self

    def predict_proba(self, X):
        valores = X.loc[:, self.variables].to_numpy(dtype=float)
        logp = np.tile(np.log(self.prior.to_numpy()), (len(X), 1))
        for j, clase in enumerate(self.clases):
            sd = self.desviaciones.loc[clase].to_numpy().copy()
            sd[sd <= 0] = self.umbral
            densidad = stats.norm.pdf(valores, loc=self.medias.loc[clase].to_numpy(), scale=sd)
            densidad[densidad <= 0] = self.umbral
            densidad[np.isnan(valores)] = 1
            logp[:, j] += np.log(densidad).sum(axis=1)
        probabilidades = np.exp(logp - logsumexp(logp, axis=1, keepdims=True))
        return pd.DataFrame(probabilidades, index=X.index, columns=self.clases)

    def predict(self, X):
        return self.predict_proba(X).idxmax(axis=1).rename("Predicción")

In [ ]:
nb = BayesGaussiano().fit(X_train_nb, y_train_nb)
nb.clases
pd.DataFrame({"Frecuencia a priori": nb.recuentos, "Probabilidad a priori": nb.prior})

In [ ]:
tablas_nb = pd.concat(nb.tablas, names=["Variable", "Clase"])
tablas_nb

#### Inspeccionar los parámetros de una variable

In [ ]:
def explorar_tabla_nb(variable):
    display(nb.tablas[variable])
    fig, ax = plt.subplots()
    medias, sd = nb.medias[variable], nb.desviaciones[variable]
    ax.bar(np.arange(len(medias)), medias, yerr=sd, capsize=3)
    ax.set(xticks=np.arange(len(medias)), xticklabels=medias.index,
           xlabel="Clase", ylabel="Media ± desviación típica", title=variable)
    plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
    fig.tight_layout()
    plt.show()

control_tabla_nb = widgets.interactive(explorar_tabla_nb,
    variable=widgets.Dropdown(options=list(predictores_nb), description="Variable"))
display(control_tabla_nb)

### Capacidad predictiva global

In [ ]:
preds_nb = nb.predict(X_test_nb)
t_nb = tabla_confusion(y_test_nb, preds_nb, nb.clases)
p_acierto = np.mean(preds_nb.to_numpy() == y_test_nb.to_numpy())
t_nb
p_acierto

### Probabilidades de clase

Cada fila suma 1 y conserva el índice del producto.
El mapa muestra los primeros 50 productos en su orden original, con las clases en columnas.
Puedes desplazarte por el test o cambiar el número de filas visibles.

In [ ]:
probs_nb = nb.predict_proba(X_test_nb)
probs_nb.head()

In [ ]:
def explorar_probabilidades(inicio, cantidad):
    tabla = probs_nb.iloc[inicio:inicio+cantidad]
    fig, ax = plt.subplots(figsize=(9, max(4, len(tabla)*0.12)))
    mapa = ax.imshow(tabla.to_numpy(), aspect="auto", vmin=0, vmax=1, cmap="viridis", interpolation="nearest")
    ax.set(xticks=np.arange(len(tabla.columns)), xticklabels=tabla.columns,
           xlabel="Clase", ylabel="Producto (orden de test)",
           title=f"Probabilidades · filas {inicio} a {inicio+len(tabla)-1}")
    posiciones = np.linspace(0, len(tabla)-1, min(10, len(tabla)), dtype=int)
    ax.set_yticks(posiciones)
    ax.set_yticklabels(tabla.index[posiciones])
    plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
    fig.colorbar(mapa, ax=ax, label="Probabilidad")
    fig.tight_layout()
    plt.show()
    display(tabla.head())

control_probabilidades = widgets.interactive(explorar_probabilidades,
    inicio=widgets.IntSlider(value=0, min=0, max=len(probs_nb)-1, description="Inicio", continuous_update=False),
    cantidad=widgets.IntSlider(value=50, min=10, max=100, step=10, description="Filas", continuous_update=False))
display(control_probabilidades)

### Acierto por clase

Normalizamos cada columna de la matriz de confusión: el denominador es el número real de
productos de esa clase. La diagonal expresa su sensibilidad (*recall*).

In [ ]:
acierto_por_clase = pd.Series(np.diag(t_nb.to_numpy()) / t_nb.sum(axis=0).to_numpy(),
                             index=nb.clases, name="Acierto por clase")
acierto_por_clase
fig, ax = plt.subplots()
ax.bar(acierto_por_clase.index, acierto_por_clase)
ax.set(ylim=(0, 1), ylabel="Proporción", title="Acierto por clase real")
plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
fig.tight_layout()
plt.show()

### Importancia de las variables mediante Gini

Calculamos el coeficiente de Gini de las nueve **medias por clase** de cada característica.
Un valor alto indica medias más desiguales entre clases. Es una medida exploratoria de separación.
La paginación permite recorrer todas las variables en paneles de nueve gráficos.

In [ ]:
def gini_medias(x):
    x = np.asarray(x, dtype=float)
    if np.any(x < 0):
        raise ValueError("El índice requiere medias no negativas.")
    return np.abs(x[:, None] - x[None, :]).sum() / (2*len(x)*x.sum()) if x.sum() else 0.0


def panel_medias(variables, titulo):
    fig, ejes = plt.subplots(3, 3, figsize=(12, 9), constrained_layout=True)
    for ax, variable in zip(ejes.flat, variables):
        medias = nb.medias[variable]
        ax.bar(np.arange(len(medias)), medias, color="steelblue")
        ax.set(title=f"{variable} · Gini {importancia[variable]:.3f}",
               xticks=np.arange(len(medias)), xticklabels=range(1, len(medias)+1),
               xlabel="Clase", ylabel="Media")
    for ax in list(ejes.flat)[len(variables):]:
        ax.set_visible(False)
    fig.suptitle(titulo)
    plt.show()

In [ ]:
importancia = nb.medias.apply(gini_medias, axis=0).rename("Gini")
importancia.sort_values(ascending=False).to_frame()

In [ ]:
def explorar_medias(pagina):
    comienzo = (pagina-1)*9
    variables = list(predictores_nb[comienzo:comienzo+9])
    panel_medias(variables, f"Medias por clase · página {pagina}")

control_medias = widgets.interactive(explorar_medias,
    pagina=widgets.IntSlider(value=1, min=1, max=int(np.ceil(len(predictores_nb)/9)),
                             description="Página", continuous_update=False))
display(control_medias)

#### Las nueve variables con mayor Gini

In [ ]:
mas_predictoras = importancia.nlargest(9).index.tolist()
panel_medias(mas_predictoras, "Mayor desigualdad entre medias de clase")

#### Las nueve variables con menor Gini

In [ ]:
menos_predictoras = importancia.nsmallest(9).index.tolist()
panel_medias(menos_predictoras, "Menor desigualdad entre medias de clase")

### Intento de mejora 1: seleccionar predictores con Gini > 0,5

In [ ]:
fig, ax = plt.subplots()
ax.hist(importancia, bins=12, edgecolor="white")
ax.axvline(0.5, color="firebrick", linestyle="--", label="Umbral 0,5")
ax.set(xlabel="Gini de las medias", ylabel="Variables", title="Distribución de la importancia")
ax.legend()
plt.show()

In [ ]:
seleccion_gini = importancia.index[importancia > 0.5].tolist()
train1 = train_nb[seleccion_gini + ["target"]]
nb1 = BayesGaussiano().fit(train1.drop(columns="target"), y_train_nb)
preds_nb1 = nb1.predict(X_test_nb)
p_acierto1 = np.mean(preds_nb1.to_numpy() == y_test_nb.to_numpy())
len(seleccion_gini)
seleccion_gini
p_acierto1

In [ ]:
def explorar_gini(umbral):
    seleccion = importancia.index[importancia > umbral].tolist()
    if not seleccion:
        display("Ninguna variable supera este umbral.")
        return
    modelo = BayesGaussiano().fit(X_train_nb[seleccion], y_train_nb)
    prediccion = modelo.predict(X_test_nb)
    display(pd.Series({"Variables seleccionadas": len(seleccion),
                       "Proporción de aciertos": np.mean(prediccion.to_numpy() == y_test_nb.to_numpy())}))
    display(importancia.loc[seleccion].sort_values(ascending=False).to_frame())

control_gini = widgets.interactive(explorar_gini,
    umbral=widgets.FloatSlider(value=0.5, min=0, max=0.9, step=0.05, description="Gini mínimo", continuous_update=False))
display(control_gini)

### Intento de mejora 2: retirar variables correlacionadas

#### Sistema 1: correlaciones positivas mayores que 0,8

Contamos, para cada característica, cuántas **otras** características superan ese umbral.
Retiramos todas las variables que tienen al menos una correlación alta.
Se reutiliza la matriz descriptiva del conjunto completo para conservar este recorrido exploratorio.
Una evaluación final requiere estimar estas selecciones solo con entrenamiento.

In [ ]:
# La diagonal (cada variable consigo misma) no cuenta como correlación alta.
diagonal = np.eye(len(cor_productos), dtype=bool)
cor_altas = cor_productos.gt(0.8).mask(diagonal, False)
numero_cor_altas = cor_altas.sum(axis=0).sort_values(ascending=False)
numero_cor_altas
retiradas_cor = numero_cor_altas.index[numero_cor_altas >= 1].tolist()
train2 = train_nb.drop(columns=retiradas_cor)

In [ ]:
nb2 = BayesGaussiano().fit(train2.drop(columns="target"), y_train_nb)
preds_nb2 = nb2.predict(X_test_nb)
p_acierto2 = np.mean(preds_nb2.to_numpy() == y_test_nb.to_numpy())
retiradas_cor
p_acierto2

#### Sistema 2: suma de correlaciones

Sumamos las correlaciones de cada columna y retiramos las primeras 30 variables del orden descendente.
La suma conserva el signo de las correlaciones.

In [ ]:
suma_cor = cor_productos.sum(axis=0).sort_values(ascending=False)
suma_cor
retiradas_suma = suma_cor.head(30).index.tolist()
train3 = train_nb.drop(columns=retiradas_suma)
nb3 = BayesGaussiano().fit(train3.drop(columns="target"), y_train_nb)
preds_nb3 = nb3.predict(X_test_nb)
p_acierto3 = np.mean(preds_nb3.to_numpy() == y_test_nb.to_numpy())
p_acierto3

#### Sistema 3: R² al explicar cada predictor con los restantes

En cada iteración calculamos el R² de cada variable frente a todas las demás y retiramos la que
tiene el mayor valor. Repetimos mientras el máximo calculado sea al menos 0,5.
La última vuelta también retira una variable antes de comprobar la condición de salida.

La función usa la matriz de correlaciones para obtener los R² de las regresiones con intercepto,
evitando repetir los ajustes sobre todas las filas. `historial_r2` registra cada retirada.

In [ ]:
def r2_multiples(correlaciones):
    """R² de cada variable explicada por todas las demás, con intercepto."""
    matriz = np.asarray(correlaciones, dtype=float)
    if len(matriz) == 1:
        return pd.Series([0.0], index=correlaciones.index)
    if np.linalg.matrix_rank(matriz) == len(matriz):
        valores = 1 - 1 / np.diag(np.linalg.inv(matriz))
    else:
        valores = []
        for j in range(len(matriz)):
            otras = np.arange(len(matriz)) != j
            r = matriz[j, otras]
            valores.append(r @ np.linalg.pinv(matriz[np.ix_(otras, otras)]) @ r)
    return pd.Series(np.clip(valores, 0, 1), index=correlaciones.index, name="R²")


def reducir_por_r2(X, limite=0.5):
    correlaciones = X.corr()
    historial = []
    maximo = limite
    while maximo >= limite and len(correlaciones) > 1:
        valores = r2_multiples(correlaciones)
        variable = valores.idxmax()
        maximo = valores[variable]
        correlaciones = correlaciones.drop(index=variable, columns=variable)
        historial.append({"Iteración": len(historial)+1, "Retirada": variable,
                          "R² máximo": maximo, "Variables restantes": len(correlaciones)})
    return list(correlaciones.columns), pd.DataFrame(historial)

In [ ]:
with threadpool_limits(limits=1):
    seleccion_r2, historial_r2 = reducir_por_r2(X_train_nb, limite=0.5)

historial_r2
len(seleccion_r2)

In [ ]:
train4 = train_nb[seleccion_r2 + ["target"]]
test4 = test_nb[train4.columns]
nb4 = BayesGaussiano().fit(train4.drop(columns="target"), y_train_nb)
preds_nb4 = nb4.predict(test4.drop(columns="target"))
t_nb4 = tabla_confusion(y_test_nb, preds_nb4, nb.clases)
p_acierto4 = np.mean(preds_nb4.to_numpy() == y_test_nb.to_numpy())
t_nb4
p_acierto4

### Intento de mejora 3: corrección de Laplace

Laplace suaviza tablas de frecuencias categóricas. Aquí todas las características entran como
numéricas y se describen mediante gaussianas, de modo que el parámetro no modifica el ajuste.
Lo dejamos explícito en la interfaz del modelo para poder inspeccionarlo.

In [ ]:
nb5 = BayesGaussiano(laplace=0.1).fit(train3.drop(columns="target"), y_train_nb)
preds_nb5 = nb5.predict(X_test_nb)
p_acierto5 = np.mean(preds_nb5.to_numpy() == y_test_nb.to_numpy())
nb5.laplace
p_acierto5

### Intento de mejora 4: densidades kernel

Estimamos una densidad gaussiana univariante para cada característica y clase.
El ancho de banda usa la regla de Silverman con factor 0,9 y se adapta a la dispersión de cada grupo.
Para mantener ágil el ajuste se emplean binning lineal, convolución FFT y una rejilla de 512 puntos.
Las probabilidades se calculan interpolando esas densidades.

In [ ]:
def densidad_kernel(x, puntos=512):
    """KDE gaussiana univariante: binning lineal, convolución FFT e interpolación."""
    x = np.asarray(x, dtype=float)
    sd = x.std(ddof=1)
    iqr = np.subtract(*np.percentile(x, [75, 25])) / 1.34
    escala = min(sd, iqr)
    if escala <= 0:
        escala = sd if sd > 0 else (abs(x[0]) if x[0] != 0 else 1)
    ancho = 0.9 * escala * len(x)**(-0.2)
    desde, hasta = x.min()-3*ancho, x.max()+3*ancho
    grilla = np.linspace(desde-4*ancho, hasta+4*ancho, puntos)
    paso = grilla[1]-grilla[0]
    coordenadas = (x-grilla[0]) / paso
    inferiores = np.floor(coordenadas).astype(int)
    fraccion = coordenadas-inferiores
    masas = np.bincount(inferiores, weights=1-fraccion, minlength=puntos)
    masas += np.bincount(inferiores+1, weights=fraccion, minlength=puntos)
    desplazamientos = np.arange(-(puntos-1), puntos)*paso
    nucleo = stats.norm.pdf(desplazamientos, scale=ancho)
    densidad = np.maximum(fftconvolve(masas/len(x), nucleo, mode="same"), 0)
    salida = np.linspace(desde, hasta, puntos)
    return SimpleNamespace(x=salida, y=np.interp(salida, grilla, densidad), ancho=ancho)


class BayesKernel:
    """Una densidad univariante por predictor y clase, con probabilidades a priori."""

    def __init__(self, puntos=512, umbral=0.001):
        self.puntos, self.umbral = puntos, umbral

    def fit(self, X, y):
        self.variables = list(X.columns)
        self.clases = np.unique(y)
        recuentos = pd.Series(y).value_counts().reindex(self.clases)
        self.prior = recuentos / recuentos.sum()
        self.densidades = {v: {} for v in self.variables}
        for clase in self.clases:
            grupo = X.loc[np.asarray(y) == clase]
            for v in self.variables:
                self.densidades[v][clase] = densidad_kernel(grupo[v], self.puntos)
        return self

    def predict_proba(self, X):
        logp = np.tile(np.log(self.prior.to_numpy()), (len(X), 1))
        for variable in self.variables:
            valores = X[variable].to_numpy(dtype=float)
            for j, clase in enumerate(self.clases):
                d = self.densidades[variable][clase]
                densidad = np.interp(valores, d.x, d.y)
                densidad[densidad <= 0] = self.umbral
                densidad[np.isnan(valores)] = 1
                logp[:, j] += np.log(densidad)
        return pd.DataFrame(np.exp(logp-logsumexp(logp, axis=1, keepdims=True)),
                            index=X.index, columns=self.clases)

    def predict(self, X):
        return self.predict_proba(X).idxmax(axis=1).rename("Predicción")

In [ ]:
nb6 = BayesKernel(puntos=512).fit(X_train_nb, y_train_nb)
preds_nb6 = nb6.predict(X_test_nb)
t_nb6 = tabla_confusion(y_test_nb, preds_nb6, nb.clases)
p_acierto6 = np.mean(preds_nb6.to_numpy() == y_test_nb.to_numpy())
t_nb6
p_acierto6

#### Inspeccionar una densidad

Comenzamos con `feat_45` en el intervalo de −1 a 1 y una línea de referencia en cero.
Puedes elegir otra característica, una clase concreta o ver todo el rango.

In [ ]:
def explorar_densidad(variable, cerca_de_cero, clase):
    clases = nb6.clases if clase == "Todas" else [clase]
    fig, ax = plt.subplots(figsize=(9, 4))
    for etiqueta in clases:
        densidad = nb6.densidades[variable][etiqueta]
        ax.plot(densidad.x, densidad.y, label=etiqueta)
    ax.axvline(0, color="black", linewidth=0.8)
    if cerca_de_cero:
        ax.set_xlim(-1, 1)
        # Ajustar la altura al tramo visible facilita leer el detalle alrededor de cero.
        alturas = [np.interp(np.linspace(-1, 1, 200), nb6.densidades[variable][c].x,
                            nb6.densidades[variable][c].y).max() for c in clases]
        ax.set_ylim(0, max(alturas)*1.08 if max(alturas) > 0 else 1)
    ax.set(xlabel=variable, ylabel="Densidad", title="Distribución condicional por clase")
    ax.legend(ncol=3, fontsize=8)
    fig.tight_layout()
    plt.show()
    display(pd.Series({c: nb6.densidades[variable][c].ancho for c in clases}, name="Ancho de banda"))

control_densidad = widgets.interactive(explorar_densidad,
    variable=widgets.Dropdown(options=list(predictores_nb), value="feat_45", description="Variable"),
    cerca_de_cero=widgets.Checkbox(value=True, description="Detalle alrededor de 0"),
    clase=widgets.Dropdown(options=["Todas"] + list(nb6.clases), description="Clase"))
display(control_densidad)

## Ejercicio 6-2

1. Reduce el número de variables predictoras a **20** y aplica KNN con diferentes números de vecinos.
   ¿Mejora la capacidad predictiva medida mediante la proporción de aciertos?

In [ ]:
# 1. Seleccionar 20 predictores, ajustar KNN y estudiar el acierto.

2. Obtén probabilidades de clase con KNN y promédialas con las de Naive Bayes.
   Asigna cada producto a la clase con mayor probabilidad media.
   ¿Mejora la proporción de aciertos? Tiempo orientativo: 15 minutos.

Conserva la misma partición de entrenamiento y test, y alinea filas y clases antes de promediar probabilidades.

In [ ]:
# 2. Promediar las probabilidades de KNN y Naive Bayes y evaluar la asignación.